# Run the GraphSAGE baseline and the structural optimizer on any RelBench task

Pick a task in the first cell. The notebook trains both models with the settings tuned on
rel-f1 driver-top3, saves every run's results and trained weights, and compares them with the
paper's number.

| Name | Meaning |
|---|---|
| **GraphSAGE baseline** | RelBench's standard GNN model |
| **Structural optimizer** | the paper's method: the baseline plus table and column switches, added edges, edge-type switches and a bottleneck |
| **Full-history test** | the paper's test setup: a prediction at time *t* may use every row before *t* |
| **RelBench cutoff test** | RelBench's standard setup: test predictions use only rows before the test period |

**Steps:**
1. *Runtime → Change runtime type → T4 GPU* (or better). For the larger databases, also turn
   on *High-RAM*.
2. Choose the task and options in the first cell.
3. *Runtime → Run all*. Google Drive asks for access once.
4. If Colab disconnects, run all again. Finished runs are kept in Drive and skipped.
5. Send back the zip from the last cell.

**Settings used** (tuned on rel-f1 driver-top3): 1 GNN layer, hidden size 128, 64 sampled
neighbours, learning rate 1e-3, weight decay 5e-3, batch 256, 50 epochs. The structural
optimizer also uses switch penalty 1e-4, bottleneck weight 1e-3, KNN 10, the previous 5
records for temporal edges, and a test-time cut-off chosen on validation.


In [ ]:
# @title Settings
TASK = 'rel-f1 / driver-top3'  # @param ['rel-f1 / driver-top3', 'rel-f1 / driver-dnf', 'rel-trial / study-outcome', 'rel-avito / user-visits', 'rel-avito / user-clicks', 'rel-event / user-repeat', 'rel-event / user-ignore', 'rel-stack / user-engagement', 'rel-stack / user-badge', 'rel-amazon / user-churn', 'rel-amazon / item-churn', 'rel-hm / user-churn']
TEST_SETUP = 'full-history'  # @param ['full-history', 'relbench-cutoff']
RUN_BASELINE = True  # @param {type:'boolean'}
RUN_OPTIMIZER = True  # @param {type:'boolean'}
SEEDS = '0,1,2'  # @param {type:'string'}
EPOCHS = 50  # @param {type:'integer'}
MAX_STEPS_PER_EPOCH = 2000  # @param {type:'integer'}
SAVE_CHECKPOINTS = True  # @param {type:'boolean'}
RUN_QUICK_CHECK = True  # @param {type:'boolean'}
USE_DRIVE = True  # @param {type:'boolean'}
CLEAR_CACHE = False  # @param {type:'boolean'}
DRIVE_DIR = '/content/drive/MyDrive/cs224w-structural-optimizer'  # @param {type:'string'}

# Code versions (change PROJECT_BRANCH to 'main' once step/report is merged):
PROJECT_REPO = 'https://github.com/HARSHALK2598/cs224w-structural-optimizer'
PROJECT_BRANCH = 'step/report'
PYG_REPO = 'https://github.com/HARSHALK2598/pytorch_geometric'
PYG_BRANCH = 'contrib/structural-optimizer'

DATASET, TASK_NAME = [s.strip() for s in TASK.split('/')]
print('task:', DATASET, TASK_NAME, '| test setup:', TEST_SETUP)


## 1. Environment

On Colab this clones both repositories and installs pinned versions: torch 2.12.0 (CUDA 12.6
build on a GPU), pyg-lib 0.9.0, the PyG fork, RelBench 3.1.0, PyTorch Frame 0.3.0 and
sentence-transformers. Training runs in a separate process, so no restart is needed.

With `USE_DRIVE`, the prepared graphs (including text features, the slowest step) are cached in
Drive, so a second session on the same task starts much faster. Downloads stay on Colab's
local disk. If a run fails while reading a cached file (for example after a disconnect during
the first session), set `CLEAR_CACHE = True` once to rebuild it.


In [ ]:
import os, subprocess, sys, shutil
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules


def sh(cmd, cwd=None):
    """Runs a shell command, streaming its output; stops on failure."""
    print('$', cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed ({proc.returncode}): {cmd}')


HAS_GPU = shutil.which('nvidia-smi') is not None and subprocess.run(
    ['nvidia-smi'], capture_output=True).returncode == 0
if IN_COLAB and not HAS_GPU:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run all again.')

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if IN_COLAB:
    ROOT = Path('/content/cs224w-structural-optimizer')
    if not ROOT.exists():
        sh(f'git clone -q --branch {PROJECT_BRANCH} {PROJECT_REPO} {ROOT}')
    else:
        sh(f'git -C {ROOT} pull -q origin {PROJECT_BRANCH}')
    pyg = ROOT / 'pytorch_geometric'
    if not pyg.exists():
        sh(f'git clone -q --branch {PYG_BRANCH} {PYG_REPO} {pyg}')
    else:
        sh(f'git -C {pyg} pull -q origin {PYG_BRANCH}')

    build = 'cu126' if HAS_GPU else 'cpu'
    pip = f'{sys.executable} -m pip install -q'
    sh(f'{sys.executable} -m pip uninstall -q -y torchvision torchaudio')
    sh(f'{pip} torch==2.12.0 --index-url https://download.pytorch.org/whl/{build}')
    sh(f'{pip} pyg_lib==0.9.0 -f https://data.pyg.org/whl/torch-2.12.0+{build}.html')
    sh(f'{pip} -e {pyg}')
    sh(f'{pip} relbench==3.1.0 pytorch-frame==0.3.0 sentence-transformers==6.3.0 pyyaml pandas')

    # Downloads (data/hf) stay on Colab's local disk: Drive's file mount
    # breaks the download cache. Only prepared graphs are kept in Drive.
    data_dir = ROOT / 'data'
    if data_dir.is_symlink():  # Left by an earlier version of this notebook.
        data_dir.unlink()
    data_dir.mkdir(exist_ok=True)
    old = Path(DRIVE_DIR) / 'cache' / 'data'
    if USE_DRIVE and old.exists():  # That version's cache, possibly corrupt.
        print('removing the old cache folder', old)
        shutil.rmtree(old, ignore_errors=True)
    if USE_DRIVE:
        cache = Path(DRIVE_DIR) / 'cache' / 'materialized'
        if CLEAR_CACHE:
            shutil.rmtree(cache, ignore_errors=True)
        cache.mkdir(parents=True, exist_ok=True)
        link = data_dir / 'materialized'
        if link.is_symlink():
            link.unlink()
        elif link.exists():
            shutil.rmtree(link)
        link.symlink_to(cache)
    elif CLEAR_CACHE:
        shutil.rmtree(data_dir / 'materialized', ignore_errors=True)
else:
    ROOT = Path(os.environ.get('PROJECT_ROOT', Path.cwd())).resolve()
    while not (ROOT / 'experiments' / 'sweep.py').exists():
        if ROOT.parent == ROOT:
            raise RuntimeError('Start the notebook inside the project repository.')
        ROOT = ROOT.parent

os.chdir(ROOT)
print('project root:', ROOT, '| Colab:', IN_COLAB, '| GPU:', HAS_GPU)
sh(f'{sys.executable} -c "import torch, torch_geometric, relbench; '
   f'print(\'torch\', torch.__version__, \'| cuda\', torch.cuda.is_available(), '
   f'torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'\', '
   f'\'| pyg\', torch_geometric.__version__, \'| relbench\', relbench.__version__)"')
sh('git log --oneline -1')


## 2. The runs

This cell writes the run plan to `config.yaml` in the results folder and prints it. Results go to
`<DRIVE_DIR>/datasets/<dataset>_<task>_<test setup>/`.


In [ ]:
import yaml

seeds = [int(s) for s in SEEDS.split(',') if s.strip()]
fixed = dict(dataset=DATASET, task=TASK_NAME, epochs=EPOCHS, lr=0.001,
             weight_decay=0.005, batch_size=256, num_neighbors=64,
             channels=128, num_layers=1, temporal_strategy='uniform',
             max_steps_per_epoch=MAX_STEPS_PER_EPOCH, dedup_encoder=True,
             nan_safe_numerical=True)
if TEST_SETUP == 'full-history':
    fixed.update(full_db=True, strict_time=True)

arms = []
if RUN_BASELINE:
    arms.append(dict(model='base'))
if RUN_OPTIMIZER:
    arms.append(dict(model='filter', grid=dict(lambda_all=[0.0001]), fixed=dict(
        inject='temporal,twohop,knn,behavioral', template_gates=True,
        beta_vib=0.001, knn_n=10, temporal_m=5, behavioral_threshold=0.8,
        gate_init=2.0, gate_thresholds='0.5,0.7,0.78,0.82,0.86,0.9')))
assert arms, 'Choose at least one model.'

NAME = f'{DATASET}_{TASK_NAME}_{TEST_SETUP}'
CFG = dict(name=NAME, fixed=fixed, seeds=seeds, arms=arms)
base_dir = Path(DRIVE_DIR) if (IN_COLAB and USE_DRIVE) else ROOT / 'experiments' / 'results'
OUT = base_dir / 'datasets' / NAME
OUT.mkdir(parents=True, exist_ok=True)
CONFIG = OUT / 'config.yaml'
CONFIG.write_text(yaml.safe_dump(CFG, sort_keys=False))
print('results folder:', OUT)
print(CONFIG.read_text())


## 3. Quick check

One run of each model for one short epoch (20 steps), in a separate `check/` folder. The first time
on a task this also downloads the database and prepares the graph, which can take from a few
minutes (rel-f1) to much longer for the large databases.


In [ ]:
if RUN_QUICK_CHECK:
    check = OUT / 'check'
    shutil.rmtree(check, ignore_errors=True)
    check_cfg = dict(CFG, name=NAME + '_check', seeds=seeds[:1],
                     fixed=dict(fixed, max_steps_per_epoch=20))
    (OUT / 'check_config.yaml').write_text(yaml.safe_dump(check_cfg, sort_keys=False))
    sh(f'{sys.executable} experiments/sweep.py "{OUT / "check_config.yaml"}" '
       f'--epochs 1 --out "{check}"')


## 4. Train

One line per finished run: validation and test score, best epoch, open switches and time. Running
this cell again continues where it stopped. With `SAVE_CHECKPOINTS`, each run's best-validation
weights go to `checkpoints/<run>.pt`, together with its settings, chosen cut-off and scores.


In [ ]:
extra = ' --save-checkpoints' if SAVE_CHECKPOINTS else ''
sh(f'{sys.executable} experiments/sweep.py "{CONFIG}" --out "{OUT}"{extra}')


## 5. Results

Mean ± std over seeds, as ROC-AUC × 100. The structural optimizer's cut-off is chosen on
validation; test is read once per run.


In [ ]:
import pandas as pd

PAPER = {  # The paper's structural optimizer, test ROC-AUC:
    ('rel-trial', 'study-outcome'): 72.35, ('rel-f1', 'driver-dnf'): 78.21,
    ('rel-f1', 'driver-top3'): 86.82, ('rel-event', 'user-repeat'): 82.36,
    ('rel-event', 'user-ignore'): 81.25, ('rel-avito', 'user-clicks'): 67.53,
    ('rel-avito', 'user-visits'): 70.15, ('rel-stack', 'user-engagement'): 91.62,
    ('rel-stack', 'user-badge'): 90.47, ('rel-amazon', 'item-churn'): 84.17,
}
NAMES = {'base': 'GraphSAGE baseline', 'filter': 'Structural optimizer'}

res = pd.read_csv(OUT / 'results.csv')
metric = res['tune_metric'].iloc[0]
def mean_std(x):
    x = 100 * x
    return f'{x.mean():.2f}' if len(x) == 1 else f'{x.mean():.2f} ± {x.std():.2f}'


rows = []
for model, g in res.groupby('model'):
    row = {'model': NAMES.get(model, model), 'runs': len(g),
           'validation': mean_std(g[f'val_{metric}']),
           'test': mean_std(g[f'test_{metric}']),
           'minutes per run': round(g['minutes'].mean(), 1)}
    if model == 'filter':
        row.update({'cut-offs chosen': ', '.join(f'{t:g}' for t in g['gate_threshold']),
                    'open tables': ', '.join(str(int(v)) for v in g['open_tables']),
                    'open added edge types': ', '.join(str(int(v)) for v in g['open_templates'])})
    rows.append(row)
print(f'{DATASET} / {TASK_NAME}, {TEST_SETUP} test, device {res["device"].iloc[0]}')
print(pd.DataFrame(rows).to_string(index=False))
paper = PAPER.get((DATASET, TASK_NAME))
print('\nPaper, structural optimizer:', paper if paper is not None else 'not listed')


## 6. Reload a checkpoint (optional)

Rebuilds one saved model and scores it on test again. Scores can differ slightly from the saved
ones, because the test-time neighbour sampler draws random neighbours.


In [ ]:
RELOAD = False  # @param {type:'boolean'}
if RELOAD:
    import torch
    sys.path.insert(0, str(ROOT / 'experiments'))
    import train, sweep
    path = sorted((OUT / 'checkpoints').glob('*.pt'))[-1]  # or pick a file
    ck = torch.load(path, weights_only=False)
    skip = {'config', 'smoke', 'results'}
    args = train.parse_args(sweep.to_argv({k: v for k, v in ck['args'].items() if k not in skip}))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dataset = train.load_dataset(args.dataset)
    task = dataset.load_task(args.task)
    data, col_stats = train.load_graph(dataset, task, args.text_embedder, device, args.full_db)
    data = train.prepare_graph(data, args)
    spec = train.task_spec(task)
    model = train.build_model(args, data, col_stats, spec.out_channels, task.entity_table).to(device)
    model.load_state_dict(ck['model'])
    if ck['gate_threshold'] is not None:
        train.set_gate_threshold(model, ck['gate_threshold'])
    pred = train.predict(model, train.make_loaders(data, task, args)['test'], spec,
                         task.entity_table, device)
    print(path.name)
    print('test now:', train.evaluate(task, spec, pred, 'test'), '| saved:', ck['test'])


## 7. Download

Writes `<name>.zip` with everything in the results folder: `config.yaml`, `results.csv`,
`summary.csv`, `env.json`, `logs/`, `gates/` and (if saved) `checkpoints/`. Send it back, and
save this notebook with its outputs (*File → Download → .ipynb*).


In [ ]:
import tempfile
zip_dir = Path('/content') if IN_COLAB else Path(tempfile.gettempdir())
archive = shutil.make_archive(str(zip_dir / NAME), 'zip', root_dir=OUT)
print('wrote', archive, f'({Path(archive).stat().st_size / 1e6:.1f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(archive)
